# Sentiment Classification Project - GloVe Baseline

The goal is to build a clean English-GloVe baseline, measure where it has vocabulary coverage, and use the validation results to motivate later improvements - especially regarding mutlilingual datasets.

In [1]:
from pathlib import Path
import re
import zipfile
import urllib.request

import pandas as pd
import numpy as np
from scipy import sparse

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import confusion_matrix, f1_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Load data

In [2]:
train_full = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

print(train_full.shape)
print(test_df.shape)
print(train_full.head())
print(train_full["label"].value_counts().sort_index())

(252000, 3)
(168000, 2)
   id                                           sentence  label
0   0  Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...      4
1   1  Passt wie angegossen\n\nTasche kam schnell und...      4
2   2  Schlechte Qualität\n\nIch habe sehr lange auf ...      1
3   3  Bestellung nie angekommen\n\n-5 Sterne..... Am...      0
4   4  Für mich gar nicht gut\n\nMacht das Makeup gar...      1
label
0    50400
1    50400
2    50400
3    50400
4    50400
Name: count, dtype: int64


# Clean data

Preprocessing is kept intentionally light. GloVe lookup does benefit from lowercasing and whitespace normalization, but aggressive preprocessing could remove useful sentiment cues such as punctuation, negation, and emojis.

In [3]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_full["text_clean"] = train_full["sentence"].apply(clean_text)
test_df["text_clean"] = test_df["sentence"].apply(clean_text)

train_full[["sentence", "text_clean", "label"]].head()

,sentence,text_clean,label
0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,Moderner Weihnachtsbaum in weiß Dieses Jahr wo...,4
1,Passt wie angegossen\n\nTasche kam schnell und...,Passt wie angegossen Tasche kam schnell und gu...,4
2,Schlechte Qualität\n\nIch habe sehr lange auf ...,Schlechte Qualität Ich habe sehr lange auf die...,1
3,Bestellung nie angekommen\n\n-5 Sterne..... Am...,Bestellung nie angekommen -5 Sterne..... Am 15...,0
4,Für mich gar nicht gut\n\nMacht das Makeup gar...,Für mich gar nicht gut Macht das Makeup gar ni...,1


# Simple language detection
## ToDo: Find better language classification method

This is a heuristic, not a production language detector. It is useful for analysis because the dataset is mostly German and English. We count common German/English function words and give German a small boost when umlauts or `ß` occur.


In [4]:
DE_WORDS = set("""
der die das und ist nicht ich ein eine einer einem einen zu mit für auf den im in es auch sehr aber von
sie er dem des dass habe hat war sind so wie man nur noch wenn kein keine keinen als doch nach kann zum
zur bei oder um aus
""".split())

EN_WORDS = set("""
the and is not i a an to with for on it also very but of in you this that have has was are so as only
if no my we they he she from or by at be been can do does did would could should after when what which
""".split())

WORD_RE = re.compile(r"[A-Za-zÄÖÜäöüß]+(?:'[A-Za-z]+)?")

def tokenize_words(text):
    return [token.lower() for token in WORD_RE.findall(str(text))]

def detect_language_simple(text):
    tokens = tokenize_words(text)
    de_score = sum(token in DE_WORDS for token in tokens)
    en_score = sum(token in EN_WORDS for token in tokens)
    de_score += sum(any(char in token for char in "äöüß") for token in tokens)
    
    if de_score >= en_score + 2:
        return "de"
    if en_score >= de_score + 2:
        return "en"
    return "uncertain"

train_full["lang"] = train_full["text_clean"].apply(detect_language_simple)
test_df["lang"] = test_df["text_clean"].apply(detect_language_simple)

print(train_full["lang"].value_counts(normalize=True))
pd.crosstab(train_full["label"], train_full["lang"], normalize="index").round(3)

lang
de           0.481504
en           0.478111
uncertain    0.040385
Name: proportion, dtype: float64


lang,de,en,uncertain
label,,,
0,0.487,0.480,0.032
1,0.492,0.484,0.024
2,0.487,0.485,0.027
3,0.477,0.478,0.045
4,0.464,0.463,0.073


# Build Validation Set
We use 90% of the reviews for training, and the remaining 10% for validation

In [5]:
train_df, val_df = train_test_split(
    train_full,
    test_size=0.1,
    stratify=train_full["label"],
    random_state=RANDOM_STATE,
)

Y_train = train_df["label"].to_numpy()
Y_val = val_df["label"].to_numpy()

print(train_df.shape, val_df.shape)

(226800, 5) (25200, 5)


# Load pretrained GloVe

Download `glove.6B.zip` from Stanford and extract `glove.6B.100d.txt` into `embeddings/`, or let the cell below download and extract it.

English GloVe is expected to have much weaker coverage on German reviews. We keep that limitation visible because it is one of the main things we want to analyze.

In [6]:
GLOVE_DIR = Path("embeddings")
GLOVE_ZIP = GLOVE_DIR / "glove.6B.zip"
GLOVE_PATH = GLOVE_DIR / "glove.6B.100d.txt"
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"

GLOVE_DIR.mkdir(exist_ok=True)

if not GLOVE_PATH.exists():
    if not GLOVE_ZIP.exists():
        print("Downloading GloVe. This file is large, so it may take a while.")
        urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP) as zf:
        zf.extract(GLOVE_PATH.name, path=GLOVE_DIR)

print(GLOVE_PATH)

embeddings/glove.6B.100d.txt


In [7]:
def load_glove(path):
    embeddings = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vector = np.asarray(parts[1:], dtype=np.float32)
            embeddings[word] = vector
    dim = len(next(iter(embeddings.values())))
    return embeddings, dim

glove, EMBEDDING_DIM = load_glove(GLOVE_PATH)
print(len(glove), EMBEDDING_DIM)

400000 100


# Measure vocabulary coverage

Coverage tells us how much of our review text can actually use a pretrained vector. Low coverage means the model is averaging over fewer informative words, which is especially important for German reviews when using English GloVe.

In [8]:
def coverage_report(texts, embeddings):
    total_tokens = 0
    covered_tokens = 0
    vocab = set()
    covered_vocab = set()
    
    for text in texts:
        tokens = tokenize_words(text)
        total_tokens += len(tokens)
        for token in tokens:
            vocab.add(token)
            if token in embeddings:
                covered_tokens += 1
                covered_vocab.add(token)
    
    return pd.Series({
        "token_coverage": covered_tokens / max(total_tokens, 1),
        "vocab_coverage": len(covered_vocab) / max(len(vocab), 1),
        "tokens": total_tokens,
        "vocab": len(vocab),
    })

coverage_by_language = train_df.groupby("lang")["text_clean"].apply(
    lambda texts: coverage_report(texts, glove)
).unstack()

coverage_overall = coverage_report(train_df["text_clean"], glove)

print("Overall coverage")
display(coverage_overall)
display(coverage_by_language)

Overall coverage


token_coverage    8.451420e-01
vocab_coverage    3.262246e-01
tokens            8.437293e+06
vocab             1.154450e+05
dtype: float64

,token_coverage,vocab_coverage,tokens,vocab
lang,,,,
de,0.695051,0.153827,4041845.0,86214.0
en,0.985661,0.865036,4327148.0,35891.0
uncertain,0.824671,0.624735,68300.0,8498.0


# Mean GloVe features

Each review becomes the average of all GloVe vectors found in the text. Unknown words are skipped. If no known word exists, we return a zero vector.

In [9]:
def mean_glove_vector(text, embeddings, dim):
    vectors = [embeddings[token] for token in tokenize_words(text) if token in embeddings]
    if not vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vectors, axis=0)

def build_mean_glove_matrix(texts, embeddings, dim):
    return np.vstack([mean_glove_vector(text, embeddings, dim) for text in texts])

X_train_mean = build_mean_glove_matrix(train_df["text_clean"], glove, EMBEDDING_DIM)
X_val_mean = build_mean_glove_matrix(val_df["text_clean"], glove, EMBEDDING_DIM)

print(X_train_mean.shape, X_val_mean.shape)

(226800, 100) (25200, 100)


Before training, we check whether GloVe lookup is failing completely for many reviews. Empty vectors would indicate a coverage or tokenization problem; non-empty but weak results indicate that mean pooling itself is the bottleneck.

In [10]:
train_vector_norm = np.linalg.norm(X_train_mean, axis=1)
val_vector_norm = np.linalg.norm(X_val_mean, axis=1)

print("Empty train vectors:", np.mean(train_vector_norm == 0))
print("Empty validation vectors:", np.mean(val_vector_norm == 0))
print("Train vector norm percentiles:", np.percentile(train_vector_norm, [0, 5, 25, 50, 75, 95, 100]))

diagnostics_df = train_df.copy()
diagnostics_df["vector_norm"] = train_vector_norm
diagnostics_df["is_empty_vector"] = diagnostics_df["vector_norm"] == 0

diagnostics_df.groupby("lang")[["vector_norm", "is_empty_vector"]].agg(["mean", "median"])

Empty train vectors: 0.00021164021164021165
Empty validation vectors: 0.0004365079365079365
Train vector norm percentiles: [0.         2.27110397 2.64621824 3.58249497 4.15101099 4.50188742
 7.07950783]


vector_norm           is_empty_vector       
                 mean    median            mean median
lang                                                  
de           2.698665  2.641792        0.000119    0.0
en           4.139607  4.147769        0.000000    0.0
uncertain    3.535252  3.675736        0.003818    0.0

# Train Logistic Regression on mean GloVe

Now we train a logistic regression classifier...

In [11]:
scaler_mean = StandardScaler()
X_train_mean_scaled = scaler_mean.fit_transform(X_train_mean)
X_val_mean_scaled = scaler_mean.transform(X_val_mean)

mean_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
mean_model.fit(X_train_mean_scaled, Y_train)

Y_train_pred_mean = mean_model.predict(X_train_mean_scaled)
Y_val_pred_mean = mean_model.predict(X_val_mean_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# Richer GloVe pooling

Mean pooling is very lossy. As a stronger GloVe-only representation, we concatenate the mean, standard deviation, minimum, and maximum over all word vectors in the review. This still ignores word order, but it preserves more information than a simple average.

In [12]:
def pooled_glove_vector(text, embeddings, dim):
    vectors = [embeddings[token] for token in tokenize_words(text) if token in embeddings]
    if not vectors:
        return np.zeros(dim * 4, dtype=np.float32)
    matrix = np.vstack(vectors)
    return np.concatenate([
        matrix.mean(axis=0),
        matrix.std(axis=0),
        matrix.min(axis=0),
        matrix.max(axis=0),
    ]).astype(np.float32)

def build_pooled_glove_matrix(texts, embeddings, dim):
    return np.vstack([pooled_glove_vector(text, embeddings, dim) for text in texts])

X_train_pooled = build_pooled_glove_matrix(train_df["text_clean"], glove, EMBEDDING_DIM)
X_val_pooled = build_pooled_glove_matrix(val_df["text_clean"], glove, EMBEDDING_DIM)

print(X_train_pooled.shape, X_val_pooled.shape)

(226800, 400) (25200, 400)


In [13]:
scaler_pooled = StandardScaler()
X_train_pooled_scaled = scaler_pooled.fit_transform(X_train_pooled)
X_val_pooled_scaled = scaler_pooled.transform(X_val_pooled)

pooled_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
pooled_model.fit(X_train_pooled_scaled, Y_train)

Y_train_pred_pooled = pooled_model.predict(X_train_pooled_scaled)
Y_val_pred_pooled = pooled_model.predict(X_val_pooled_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# TF-IDF weighted GloVe features

Plain averaging treats all words equally. TF-IDF weighting gives more influence to words that are informative in a review and less influence to very common words.

In [14]:
tfidf = TfidfVectorizer(tokenizer=tokenize_words, preprocessor=None, lowercase=False, min_df=2)
tfidf.fit(train_df["text_clean"])

idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
default_idf = max(tfidf.idf_)

def tfidf_weighted_glove_vector(text, embeddings, dim, idf_lookup, default_weight):
    weighted_vectors = []
    weights = []
    for token in tokenize_words(text):
        if token in embeddings:
            weight = idf_lookup.get(token, default_weight)
            weighted_vectors.append(embeddings[token] * weight)
            weights.append(weight)
    if not weighted_vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.sum(weighted_vectors, axis=0) / np.sum(weights)

def build_tfidf_glove_matrix(texts, embeddings, dim, idf_lookup, default_weight):
    return np.vstack([
        tfidf_weighted_glove_vector(text, embeddings, dim, idf_lookup, default_weight)
        for text in texts
    ])

X_train_tfidf = build_tfidf_glove_matrix(train_df["text_clean"], glove, EMBEDDING_DIM, idf, default_idf)
X_val_tfidf = build_tfidf_glove_matrix(val_df["text_clean"], glove, EMBEDDING_DIM, idf, default_idf)

print(X_train_tfidf.shape, X_val_tfidf.shape)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(226800, 100) (25200, 100)


In [15]:
scaler_tfidf = StandardScaler()
X_train_tfidf_scaled = scaler_tfidf.fit_transform(X_train_tfidf)
X_val_tfidf_scaled = scaler_tfidf.transform(X_val_tfidf)

tfidf_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
tfidf_model.fit(X_train_tfidf_scaled, Y_train)

Y_train_pred_tfidf = tfidf_model.predict(X_train_tfidf_scaled)
Y_val_pred_tfidf = tfidf_model.predict(X_val_tfidf_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# GloVe + handcrafted text features

These features capture signals that averaged embeddings ignore, such as review length, punctuation intensity, uppercase emphasis, and rough negation counts.

In [16]:
NEGATION_WORDS = set("not no never n't nicht kein keine keinen niemals ohne".split())

def handcrafted_features(text):
    text = str(text)
    tokens = tokenize_words(text)
    token_count = len(tokens)
    char_count = len(text)
    exclamation_count = text.count("!")
    question_count = text.count("?")
    uppercase_words = sum(token.isupper() and len(token) > 1 for token in re.findall(r"\b\w+\b", text))
    negation_count = sum(token in NEGATION_WORDS for token in tokens)
    known_tokens = sum(token in glove for token in tokens)
    coverage = known_tokens / max(token_count, 1)
    
    return np.array([
        np.log1p(char_count),
        np.log1p(token_count),
        exclamation_count,
        question_count,
        uppercase_words,
        negation_count,
        coverage,
    ], dtype=np.float32)

def build_handcrafted_matrix(texts):
    return np.vstack([handcrafted_features(text) for text in texts])

X_train_hand = build_handcrafted_matrix(train_df["text_clean"])
X_val_hand = build_handcrafted_matrix(val_df["text_clean"])

X_train_combined = np.hstack([X_train_tfidf, X_train_hand])
X_val_combined = np.hstack([X_val_tfidf, X_val_hand])

print(X_train_combined.shape, X_val_combined.shape)

(226800, 107) (25200, 107)


In [17]:
scaler_combined = StandardScaler()
X_train_combined_scaled = scaler_combined.fit_transform(X_train_combined)
X_val_combined_scaled = scaler_combined.transform(X_val_combined)

combined_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
combined_model.fit(X_train_combined_scaled, Y_train)

Y_train_pred_combined = combined_model.predict(X_train_combined_scaled)
Y_val_pred_combined = combined_model.predict(X_val_combined_scaled)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# BoW anchor and BoW + GloVe

The intro notebook's bag-of-words baseline is a crucial sanity check. If this reproduces the stronger score, then the data split and evaluation are fine. We then concatenate the same BoW features with GloVe features to test whether GloVe adds useful information once the lexical signal is preserved.

In [18]:
bow_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=10_000,
)

X_train_bow = bow_vectorizer.fit_transform(train_df["text_clean"])
X_val_bow = bow_vectorizer.transform(val_df["text_clean"])

bow_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
bow_model.fit(X_train_bow, Y_train)

Y_train_pred_bow = bow_model.predict(X_train_bow)
Y_val_pred_bow = bow_model.predict(X_val_bow)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [19]:
X_train_bow_glove = sparse.hstack(
    [X_train_bow, sparse.csr_matrix(X_train_combined_scaled)],
    format="csr",
)
X_val_bow_glove = sparse.hstack(
    [X_val_bow, sparse.csr_matrix(X_val_combined_scaled)],
    format="csr",
)

bow_glove_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
bow_glove_model.fit(X_train_bow_glove, Y_train)

Y_train_pred_bow_glove = bow_glove_model.predict(X_train_bow_glove)
Y_val_pred_bow_glove = bow_glove_model.predict(X_val_bow_glove)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# GloVe + lexical TF-IDF sanity check

The previous models compress each review into roughly 100 numbers. That is a very severe bottleneck for sentiment classification. This section keeps the GloVe features, but concatenates them with sparse word and character TF-IDF features. If this performs much better, then the problem is not the train/validation split or Logistic Regression itself; the problem is that averaged GloVe alone loses too much lexical information.

In [20]:
word_vectorizer = TfidfVectorizer(
    tokenizer=tokenize_words,
    preprocessor=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)

X_train_word = word_vectorizer.fit_transform(train_df["text_clean"])
X_val_word = word_vectorizer.transform(val_df["text_clean"])

X_train_char = char_vectorizer.fit_transform(train_df["text_clean"])
X_val_char = char_vectorizer.transform(val_df["text_clean"])

X_train_glove_sparse = sparse.csr_matrix(X_train_combined_scaled)
X_val_glove_sparse = sparse.csr_matrix(X_val_combined_scaled)

X_train_hybrid = sparse.hstack(
    [X_train_word, X_train_char, X_train_glove_sparse],
    format="csr",
)
X_val_hybrid = sparse.hstack(
    [X_val_word, X_val_char, X_val_glove_sparse],
    format="csr",
)

print(X_train_hybrid.shape, X_val_hybrid.shape)

/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(226800, 150107) (25200, 150107)


In [21]:
# Logistic regression trained with SGD is much faster for very large sparse text matrices.
hybrid_model = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    max_iter=30,
    tol=1e-3,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
hybrid_model.fit(X_train_hybrid, Y_train)

Y_train_pred_hybrid = hybrid_model.predict(X_train_hybrid)
Y_val_pred_hybrid = hybrid_model.predict(X_val_hybrid)

# Evaluate models

In [22]:
def competition_score(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    return  1.0 - (mae / 4.0)

def evaluate_predictions(name, y_train, y_train_pred, y_val, y_val_pred):
    return pd.Series({
        "model": name,
        "train_score": competition_score(y_train, y_train_pred),
        "val_score": competition_score(y_val, y_val_pred),
        "val_mae": mean_absolute_error(y_val, y_val_pred),
        "val_accuracy": np.mean(y_val == y_val_pred),
        "val_macro_f1": f1_score(y_val, y_val_pred, average="macro"),
    })

results = pd.DataFrame([
    evaluate_predictions("Bag-of-words + Logistic Regression", Y_train, Y_train_pred_bow, Y_val, Y_val_pred_bow),
    evaluate_predictions("Bag-of-words + GloVe + Logistic Regression", Y_train, Y_train_pred_bow_glove, Y_val, Y_val_pred_bow_glove),
    evaluate_predictions("Mean GloVe + Logistic Regression", Y_train, Y_train_pred_mean, Y_val, Y_val_pred_mean),
    evaluate_predictions("Pooled GloVe + Logistic Regression", Y_train, Y_train_pred_pooled, Y_val, Y_val_pred_pooled),
    evaluate_predictions("TF-IDF GloVe + Logistic Regression", Y_train, Y_train_pred_tfidf, Y_val, Y_val_pred_tfidf),
    evaluate_predictions("TF-IDF GloVe + Handcrafted + Logistic Regression", Y_train, Y_train_pred_combined, Y_val, Y_val_pred_combined),
    evaluate_predictions("Word/char TF-IDF + GloVe + Logistic Regression", Y_train, Y_train_pred_hybrid, Y_val, Y_val_pred_hybrid),
])

results.sort_values("val_score", ascending=False)

,model,train_score,val_score,val_mae,val_accuracy,val_macro_f1
1,Bag-of-words + GloVe + Logistic Regression,0.897082,0.869554,0.521786,0.579048,0.575853
0,Bag-of-words + Logistic Regression,0.896004,0.869355,0.522579,0.579444,0.575566
6,Word/char TF-IDF + GloVe + Logistic Regression,0.813172,0.806230,0.775079,0.488452,0.480756
3,Pooled GloVe + Logistic Regression,0.794002,0.792530,0.829881,0.449365,0.443046
5,TF-IDF GloVe + Handcrafted + Logistic Regression,0.770588,0.769315,0.922738,0.411190,0.405851
2,Mean GloVe + Logistic Regression,0.769153,0.768869,0.924524,0.414087,0.407128
4,TF-IDF GloVe + Logistic Regression,0.752205,0.752778,0.988889,0.393651,0.385315


In [23]:
val_analysis = val_df.copy()
val_analysis["pred"] = Y_val_pred_bow_glove
val_analysis["abs_error"] = (val_analysis["label"] - val_analysis["pred"]).abs()

language_scores = []
for lang, group in val_analysis.groupby("lang"):
    language_scores.append(pd.Series({
        "lang": lang,
        "n": len(group),
        "score": competition_score(group["label"], group["pred"]),
        "mae": mean_absolute_error(group["label"], group["pred"]),
        "accuracy": np.mean(group["label"] == group["pred"]),
    }))

pd.DataFrame(language_scores).sort_values("score", ascending=False)

,lang,n,score,mae,accuracy
2,uncertain,1010,0.898267,0.406931,0.681188
1,en,11938,0.870435,0.518261,0.584269
0,de,12252,0.866328,0.534688,0.565540


In [24]:
conf_matrix = confusion_matrix(Y_val, Y_val_pred_bow_glove, labels=[0,1,2,3,4])
pd.DataFrame(conf_matrix, index=[0,1,2,3,4], columns=[0,1,2,3,4])

,0,1,2,3,4
0,3525,1031,368,66,50
1,1271,2253,1198,236,82
2,445,1159,2345,910,181
3,96,219,862,2607,1256
4,63,56,153,906,3862


# Error analysis

In [25]:
very_wrong = val_analysis.sort_values("abs_error", ascending=False).head(10)

for _, row in very_wrong.iterrows():
    print("-" * 80)
    print(f"Language: {row['lang']} | Actual: {row['label']} | Predicted: {row['pred']} | Error: {row['abs_error']}")
    print(row["text_clean"][:700])

--------------------------------------------------------------------------------
Language: en | Actual: 0 | Predicted: 4 | Error: 4
These things suck Run about half the time and when you really ... These things suck Run about half the time and when you really need them because you have no power hook up for your camper they will not run in parallel. I bought 2 for them. Both have been in the shop 2 times each and still can not get them to run right. Will take them in to the shop again this week let see what happens .
--------------------------------------------------------------------------------
Language: de | Actual: 0 | Predicted: 4 | Error: 4
Nicht zu empfehlen! Ich habe bisher immer mit den one steps und dem clearblue digital und dem Monitor getestet! Jetzt waren meine one step alle und ich habe mir diese hier bestellt... Ich bin mit den one step und dem cb schon 2x schwanger geworden. Diese hier zeigen mir ein positives Ergebnis obwohl das nich gar nicht sein kann denn alle andere

# Make test submission

We use the strongest validation model from this notebook. At this stage that is usually the combined feature model, but this should be checked after running the notebook.

In [26]:
X_test_tfidf = build_tfidf_glove_matrix(test_df["text_clean"], glove, EMBEDDING_DIM, idf, default_idf)
X_test_hand = build_handcrafted_matrix(test_df["text_clean"])
X_test_combined = np.hstack([X_test_tfidf, X_test_hand])
X_test_combined_scaled = scaler_combined.transform(X_test_combined)

X_test_bow = bow_vectorizer.transform(test_df["text_clean"])
X_test_bow_glove = sparse.hstack(
    [X_test_bow, sparse.csr_matrix(X_test_combined_scaled)],
    format="csr",
)

submit_preds = bow_glove_model.predict(X_test_bow_glove)

submission = pd.DataFrame({
    "id": test_df["id"],
    "label": submit_preds,
})

Path("submissions").mkdir(exist_ok=True)
submission_path = "submissions/bow_glove_logreg_submission.csv"
submission.to_csv(submission_path, index=False)

print(submission_path)
submission.head()

submissions/bow_glove_logreg_submission.csv


,id,label
0,0,1
1,1,2
2,2,3
3,3,1
4,4,1
